# Column Dropping and Reordering for Final Datasets

This notebook:
1. Drops unwanted columns (QA flags, SWE, SWIR22, optional base GAIA/JRC point features)
2. Removes DRP censoring columns where not needed
3. Drops `nir` from all datasets
4. Reorders columns to put `month_fitted` 3rd
5. Saves and verifies all six Final Datasets in `Datasets_Ours/Final Datasets`.

In [ ]:
import pandas as pd
from pathlib import Path

FINAL_DIR = Path("Datasets_Ours/Final Datasets")
FILES = [
    "drp_training_complete.csv",
    "ec_training_complete.csv",
    "ta_training_complete.csv",
    "drp_validation.csv",
    "ec_validation.csv",
    "ta_validation.csv",
]

# Base columns to drop (if present)
DROP_COLS = {
    "qa_pixel",
    "qa_aerosol",
    "esa_processed_flag",
    "esa_observation_count",
    "esa_current_pixel_state",
    "coastal",
    "lwir11",
    "swe",     # TerraClimate
    "swir22",  # redundant vs swir16
    "nir",
}

# Drop point-based GAIA/JRC features when raster features exist
DROP_BASE_GAIA_JRC = True

# Target columns for protection / ordering
TARGET_COLS = {
    "drp": ["dissolved_reactive_phosphorus", "drp_censored_10", "drp_censored_20"],
    "ec": ["electrical_conductance"],
    "ta": ["total_alkalinity"],
}

print("Libraries imported successfully!")

In [ ]:
def infer_target_key(fname: str) -> str:
    if fname.startswith("drp_"):
        return "drp"
    if fname.startswith("ec_"):
        return "ec"
    if fname.startswith("ta_"):
        return "ta"
    return ""


def compute_drop_cols(df: pd.DataFrame, fname: str) -> list[str]:
    target_key = infer_target_key(fname)
    protected = set(TARGET_COLS.get(target_key, []))

    drop_cols = set(DROP_COLS)

    if DROP_BASE_GAIA_JRC:
        for c in df.columns:
            if c.startswith("gaia_") and not c.endswith("_1km"):
                drop_cols.add(c)
            if c.startswith("gsw_") and not c.endswith("_1km"):
                drop_cols.add(c)

    # Never drop protected targets
    drop_cols = [c for c in drop_cols if c in df.columns and c not in protected]
    return sorted(drop_cols)


def reorder_columns_with_month_fitted(df: pd.DataFrame, target_key: str) -> pd.DataFrame:
    if "month_fitted" not in df.columns:
        return df

    ordered_cols = ["latitude", "longitude", "month_fitted"]
    remaining_cols = [c for c in df.columns if c not in ordered_cols]

    target_cols = [c for c in remaining_cols if c in TARGET_COLS.get(target_key, [])]
    feature_cols = [c for c in remaining_cols if c not in target_cols]

    final_order = ordered_cols + feature_cols + target_cols
    return df[final_order]


print("Loading datasets...")
dfs = {}
for fname in FILES:
    dfs[fname] = pd.read_csv(FINAL_DIR / fname)
    print(f"  {fname}: {dfs[fname].shape}")

print("\nDropping columns and reordering...")
for fname, df in dfs.items():
    target_key = infer_target_key(fname)
    cols_to_drop = compute_drop_cols(df, fname)

    print(f"\n{fname}")
    print(f"  Original shape: {df.shape}")
    print(f"  Dropping {len(cols_to_drop)} columns:")
    print(f"  {cols_to_drop}")

    df = df.drop(columns=cols_to_drop)
    df = reorder_columns_with_month_fitted(df, target_key)

    dfs[fname] = df
    df.to_csv(FINAL_DIR / fname, index=False)

    print(f"  New shape: {df.shape}")

print("\n✓ All datasets updated and saved.")

In [ ]:
print("\nVERIFICATION")
print("=" * 60)

for fname in FILES:
    df = pd.read_csv(FINAL_DIR / fname)
    target_key = infer_target_key(fname)

    print(f"\n{fname}: {df.shape}")

    if "month_fitted" in df.columns:
        month_pos = list(df.columns).index("month_fitted")
        if month_pos == 2:
            print("  ✓ month_fitted at position 3 (3rd column)")
        else:
            print(f"  ⚠️  month_fitted at position {month_pos + 1} (expected 3)")
    else:
        print("  ⚠️  month_fitted not found")

    if "nir" in df.columns:
        print("  ⚠️  NIR still present")
    else:
        print("  ✓ NIR removed")

    # Target presence checks
    for tcol in TARGET_COLS.get(target_key, []):
        if tcol in df.columns:
            print(f"  ✓ target present: {tcol}")
        else:
            print(f"  ⚠️  target missing: {tcol}")

    # DRP censoring columns should be removed from non-DRP files
    if target_key != "drp":
        if any(c in df.columns for c in ["drp_censored_10", "drp_censored_20", "dissolved_reactive_phosphorus"]):
            print("  ⚠️  DRP-related columns still present")
        else:
            print("  ✓ DRP-related columns removed")

print("\n" + "=" * 60)
print("✓ VERIFICATION COMPLETE")